In [1]:
import pandas as pd
import numpy as np
import sklearn

print(sklearn.__version__)

1.7.2


In [3]:
pokemon = pd.read_csv("~/Downloads/pokemon_numeric.csv")

pokemon = pokemon.drop(columns=["Name"])

pokemon.head()

,Attack,Defense,Sp_Atk,Sp_Def,Speed,Height_m,Weight_kg,Catch_Rate,hasMegaEvolution
0,49,49,65,65,45,0.71,6.9,45,False
1,62,63,80,80,60,0.99,13.0,45,False
2,82,83,100,100,80,2.01,100.0,45,True
3,52,43,60,50,65,0.61,8.5,45,False
4,64,58,80,65,80,1.09,19.0,45,False


In [7]:
X = pokemon.drop(columns=["hasMegaEvolution"])

y = pokemon["hasMegaEvolution"].map({True: 1, False: 0})

print(y.value_counts())

hasMegaEvolution
0    675
1     46
Name: count, dtype: int64


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=0
)

In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [15]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(
    class_weight="balanced", y=y_train
)

In [17]:
from sklearn.metrics import confusion_matrix

modelNN = MLPClassifier(
    random_state=0,
    activation="logistic",
    solver="sgd",
    hidden_layer_sizes=8,
    max_iter=2000
)

modelNN.fit(X_train, y_train, sample_weight=sample_weights)

predicted_test = modelNN.predict(X_test)

print(confusion_matrix(y_test, predicted_test, labels=[1, 0]))

[[  8   6]
 [ 69 134]]


In [19]:
from sklearn.model_selection import GridSearchCV

modelNN = MLPClassifier(
    random_state=0,
    max_iter=2000,
    activation="logistic",
    solver="sgd"
)

params = {
    "hidden_layer_sizes": list(range(8, 21)),
    "learning_rate_init": [0.1, 0.2, 0.3, 0.4, 0.5,
                           0.6, 0.7, 0.8, 0.9, 1.0]
}

gs = GridSearchCV(
    modelNN, params, cv=3, verbose=2, scoring="recall"
)

gs.fit(X_train, y_train, sample_weight=sample_weights)

print("Best settings:", gs.best_params_)

Fitting 3 folds for each of 130 candidates, totalling 390 fits
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.1; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.1; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.1; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.2; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.2; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.2; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.3; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.3; total time=   0.1s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.3; total time=   0.0s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.4; total time=   0.1s
[CV] END .......hidden_layer_sizes=8, learning_rate_init=0.4; total time=   0.1s
[CV] END .......hidden_layer_sizes=8, learning

In [20]:
modelNN = MLPClassifier(
    random_state=0,
    max_iter=2000,
    activation="logistic",
    solver="sgd",
    hidden_layer_sizes=gs.best_params_["hidden_layer_sizes"],
    learning_rate_init=gs.best_params_["learning_rate_init"]
)

modelNN.fit(X_train, y_train, sample_weight=sample_weights)

predicted_test = modelNN.predict(X_test)

print(confusion_matrix(y_test, predicted_test, labels=[1, 0]))

[[  9   5]
 [ 62 141]]


In [42]:
from sklearn.ensemble import BaggingClassifier

modelNN = MLPClassifier(
    random_state=0,
    max_iter=2000,
    activation="logistic",
    solver="sgd",
    hidden_layer_sizes=9,
    learning_rate_init=0.1
)

In [44]:
modelBagging = BaggingClassifier(
    estimator=modelNN,
    random_state=22
)

In [49]:
from sklearn.metrics import precision_score, recall_score

modelBagging.fit(
    X_train, y_train, sample_weight=sample_weights
)

predicted_bagging = modelBagging.predict(X_test)

print(confusion_matrix(y_test, predicted_bagging, labels=[1, 0]))

print("Precision:", precision_score(y_test, predicted_bagging))
print("Recall:", recall_score(y_test, predicted_bagging))

[[  9   5]
 [ 59 144]]
Precision: 0.1323529411764706
Recall: 0.6428571428571429
